# Use Case 4: QML Time Series Forecasting (Electricity Prices)

This use case demonstrates electricity price time-series forecasting using a **Quantum Neural Network (QNN) Regressor**.

- **Author:** Jiří Tomčala (IT4Innovations National Supercomputing Center)
- **Quantum Architecture:** 8-qubit Parameterized Quantum Circuit (`ZFeatureMap` + `EfficientSU2` ansatz with 80 trainable parameters $\theta$).
- **Data:** Real historical hourly electricity market prices from OTE-CR (Czech Market Operator).
- **Environment:** Runs directly under LUMI-IQ module (`module load Lumi-IQ-qaas/VLQ`).

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np
import matplotlib.pyplot as plt
from qiskit_machine_learning.algorithms.regressors import NeuralNetworkRegressor

print("QML Time Series environment initialized.")

# Backend selection: "SIM" (fast local simulation) or "VLQ" (real QPU via LEXIS)
TARGET_BACKEND = "SIM"  # Change to "VLQ" to dispatch 1-hour ahead forecast circuit to VLQ


### 1. Load Real Historical Market Data
Load and normalize hourly electricity price series into 8-hour lookback feature windows.

In [ ]:
data_orig = np.load("data.npy", allow_pickle=True)[96:]
n = 360

min_val = min(data_orig[0:(8+n)])
max_val = max(data_orig[0:(8+n)])
data = ((2 * np.pi) * (data_orig - min_val) / (max_val - min_val)) - np.pi

inputs = []
for i in range(n):
    inputs.append(data[i:(i+8)])
inputs = np.array(inputs)

outputs = np.array(data[8:(8+n)])
outputs_norm = ((outputs - min(outputs)) / (max(outputs) - min(outputs))) - 0.5
outputs_orig = np.array(data_orig[8:(8+n)])
outputs_train_orig = outputs_orig[0:240]

print(f"Loaded {len(data_orig)} time steps. Testing window: {n} hours ahead.")


### 2. Load Pre-trained QNN Model & Perform Inference
Evaluate the 8-qubit QNN model on the test sequence.

In [ ]:
modelfile = "QNN_0.7706084854606758-model_state.dill"
from run_time_series_forecasting import load_qnn_model
reg = load_qnn_model(modelfile)

outputs_predicted_norm = reg.predict(inputs)[:, 0]
outputs_predicted_norm_pi = (outputs_predicted_norm + 0.5) * (max(outputs) - min(outputs)) + min(outputs)
outputs_predicted_orig = (outputs_predicted_norm_pi + np.pi)/(2*np.pi) * (max_val - min_val) + min_val

score = reg.score(inputs, outputs_norm)
print(f"Forecasting Accuracy Score: {score:.4f} ({score*100:.2f}%)")


### 2b. Hardware Execution on VLQ QPU (IT4Innovations)
If , encode the latest 8-hour price window into the 8-qubit parameterized circuit, transpile for VLQ native gates (CZ, PRX), submit to the QPU, and measure $\langle Z_0 \rangle$.

In [ ]:
vlq_predicted_price = None

if TARGET_BACKEND == "VLQ":
    sys.path.insert(0, os.path.abspath("../.."))
    from config import LEXIS_PROJECT, LEXIS_RESOURCE, get_lexis_token
    from qaas.client import QProvider
    from qiskit import transpile

    print("=== Initializing Connection to VLQ Quantum Computer ===")
    token = get_lexis_token()
    provider = QProvider(lexis_project=LEXIS_PROJECT, token=token)
    backend = provider.get_backend(LEXIS_RESOURCE)
    print(f"Connected to QPU: {backend.name} (target: {LEXIS_RESOURCE})")

    latest_input = inputs[-1]
    eval_circuit = reg.neural_network.circuit.copy()
    param_dict = {p: val for p, val in zip(reg.neural_network.input_params, latest_input)}
    param_dict.update({p: val for p, val in zip(reg.neural_network.weight_params, reg.weights)})
    eval_circuit.assign_parameters(param_dict, inplace=True)
    eval_circuit.measure_all()

    print("Transpiling 8-qubit QNN circuit for VLQ Star-topology native gates (CZ, PRX)...")
    t_circuit = transpile(eval_circuit, backend=backend, optimization_level=1)
    print(f"Transpiled circuit depth: {t_circuit.depth()}, CZ gates: {t_circuit.count_ops().get('cz', 0)}")

    shots = 1024
    print(f"Submitting QNN forward pass to VLQ ({shots} shots)...")
    job = backend.run(t_circuit, shots=shots)
    print(f"Job ID: {job.job_id()} queued. Waiting for measurements...")
    result = job.result(timeout_secs=600)
    counts = result.get_counts()
    print("[VLQ SUCCESS] Measurements received from VLQ QPU!")

    c0 = sum(cnt for b, cnt in counts.items() if b[-1] == "0")
    c1 = sum(cnt for b, cnt in counts.items() if b[-1] == "1")
    exp_val = (c0 - c1) / shots
    vlq_norm_pi = (exp_val + 0.5) * (max(outputs) - min(outputs)) + min(outputs)
    vlq_predicted_price = (vlq_norm_pi + np.pi) / (2 * np.pi) * (max_val - min_val) + min_val
    actual_price = outputs_orig[-1]
    print("-" * 60)
    print(f"VLQ Measured <Z_0>             : {exp_val:+.4f}")
    print(f"VLQ 1-Hour Ahead Forecast Price: {vlq_predicted_price:.2f} EUR/MWh")
    print(f"Actual Market Clearing Price   : {actual_price:.2f} EUR/MWh")
    print("-" * 60)


### 3. Forecast Plot
Compare real historical prices against the QNN prediction.

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(range(len(outputs_orig)), outputs_orig, "r--", label="Real Market Prices (OTE-CR)")
plt.plot(range(len(outputs_train_orig)), outputs_train_orig, "b.", label="Training Window")
plt.plot(range(len(outputs_predicted_orig)), outputs_predicted_orig, "g-", linewidth=2, label="QNN Forecast (8-qubit PQC)")
if vlq_predicted_price is not None:
    plt.scatter([len(outputs_orig)-1], [vlq_predicted_price], color="gold", edgecolor="black", s=120, zorder=5, label=f"VLQ QPU Forecast: {vlq_predicted_price:.2f} EUR/MWh")
plt.title(f"Electricity Price 1-Hour Ahead Forecasting (Average Precision: {score*100:.2f}%)", fontsize=13, fontweight="bold")
plt.xlabel("Time [Hours]", fontsize=11)
plt.ylabel("Price [EUR/MWh]", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("result_time_series.png", dpi=150)
plt.show()
